# MFRouter - Training

This notebook demonstrates how to train the **MFRouter** (Matrix Factorization Router).

## Overview

MFRouter uses matrix factorization to learn latent representations for both queries and LLMs,
then predicts the best LLM based on the similarity in the latent space.

**Key Features**:
- Learns embeddings for both queries and models
- Can capture collaborative filtering patterns
- Effective for large query-model interaction matrices

## 1. Environment Setup

In [ ]:
# Install required packages (for Colab)

!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.2/236.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 100.6 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires grpcio>=1.71.2, but you have grpcio 1.67.1 which is incompatible.
Cloning into 'LLMRou

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.mfrouter import MFRouter, MFRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

MFRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `latent_dim` | Dimension of latent space | 128 |
| `text_dim` | Query embedding dimension | 768 |
| `lr` | Learning rate | 0.001 |
| `epochs` | Training epochs | 5 |
| `noise_alpha` | Noise for regularization | 0.0 |

In [ ]:
import yaml

CONFIG_PATH = "configs/model_config_train/mfrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  query_embedding_data: data/example_data/routing_data/query_embeddings_longformer.pt
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
hparam:
  batch_size: 64
  epochs: 5
  latent_dim: 128
  lr: 0.001
  noise_alpha: 0.0
  text_dim: 768
metric:
  weights:
    cost: 0
    llm_judge: 0
    performance: 1
model_path:
  ini_model_path: ''
  save_model_path: saved_models/mfrouter/mfrouter.pkl



## 3. Initialize Router

In [ ]:
router = MFRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"LLM candidates: {list(router.llm_data.keys())}")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router initialized successfully!
Number of training samples: 50544
Number of LLM candidates: 14
LLM candidates: ['qwen2.5-7b-instruct', 'codegemma-7b', 'gemma-2-9b-it', 'llama-3.1-8b-instruct', 'llama3-chatqa-1.5-8b', 'mistral-7b-instruct-v0.3', 'llama-3.3-nemotron-super-49b-v1', 'llama-3.1-nemotron-51b-instruct', 'llama3-chatqa-1.5-70b', 'llama3-70b-instruct', 'mixtral-8x7b-instruct-v0.1', 'mixtral-8x22b-instruct-v0.1', 'palmyra-creative-122b', 'mistral-nemo-12b-instruct']


## 4. Training

In [ ]:
trainer = MFRouterTrainer(router=router, device='cpu')

print("Trainer initialized!")
print(f"Save path: {trainer.save_model_path}")

[MFRouterTrainer] Initialized with precomputed embeddings (fast mode).
[MFRouterTrainer] Batch size: 64
Trainer initialized!
Save path: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl


In [ ]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

Starting training...
[MFRouterTrainer] Training with 44928 samples, 702 batches per epoch
[MFRouterTrainer] Epoch 1/5 - Loss=0.380405
[MFRouterTrainer] Epoch 2/5 - Loss=0.324342
[MFRouterTrainer] Epoch 3/5 - Loss=0.306890
[MFRouterTrainer] Epoch 4/5 - Loss=0.292752
[MFRouterTrainer] Epoch 5/5 - Loss=0.281824
Successfully saved pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
[MFRouterTrainer] Model saved to /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
Training completed!


## 5. Model Verification

In [ ]:
from llmrouter.utils import load_model
import torch

saved_model = load_model(trainer.save_model_path)

print("Model loaded successfully!")
print(f"Model type: {type(saved_model).__name__}")

Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
Model loaded successfully!
Model type: OrderedDict


In [ ]:
# Test prediction
test_query = {"query": "What is the capital of France?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Input ids are automatically padded to be a multiple of `config.attention_window`: 512


model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Test query: What is the capital of France?
Routed to: qwen2.5-7b-instruct


## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up MFRouter with YAML configuration
2. **Trained Model**: Used MFRouterTrainer to learn latent representations
3. **Verified Model**: Tested routing with sample queries

**Next Steps**:
- Use `02_mfrouter_inference.ipynb` for inference
- Experiment with different latent dimensions

# MFRouter - Inference

This notebook demonstrates how to use a trained **MFRouter** for inference.

## 1. Environment Setup

In [ ]:
from llmrouter.models.mfrouter import MFRouter
from llmrouter.utils import setup_environment, load_model
import yaml

setup_environment()

## 2. Load Trained Router

In [ ]:
CONFIG_PATH = "configs/model_config_train/mfrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

config['model_path']['load_model_path'] = config['model_path']['save_model_path']

INFERENCE_CONFIG_PATH = "configs/model_config_test/mfrouter_inference.yaml"
os.makedirs(os.path.dirname(INFERENCE_CONFIG_PATH), exist_ok=True)

with open(INFERENCE_CONFIG_PATH, 'w') as f:
    yaml.dump(config, f)

router = MFRouter(yaml_path=INFERENCE_CONFIG_PATH)
print(f"Router loaded with {len(router.llm_data)} LLM candidates")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router loaded with 14 LLM candidates


## 3. Query Routing

In [ ]:
EXAMPLE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "Solve the equation: 2x + 5 = 15"},
    {"query": "Write a Python function to check if a number is prime."},
    {"query": "Explain quantum computing in simple terms."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

Routing Results:
Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
1. What is the capital of France?...
   Routed to: qwen2.5-7b-instruct
Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
2. Solve the equation: 2x + 5 = 15...
   Routed to: qwen2.5-7b-instruct
Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
3. Write a Python function to check if a number is pr...
   Routed to: llama-3.1-8b-instruct
Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
4. Explain quantum computing in simple terms....
   Routed to: qwen2.5-7b-instruct


## 4. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/mfrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")

    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")

    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)

    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl
Successfully loaded pickle model: /content/LLMRouter/llmrouter/saved_models/mfrouter/mfrouter.pkl
Routed 10 queries
Results saved to: outputs/mfrouter_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered... -> qwen2.5-7b-instruct
  2. Q: There are 3 houses in a row, numbered... -> gemma-2-9b-it
  3. Q: There are 3 houses in a row, numbered... -> gemma-2-9b-it


## Summary

This notebook demonstrated:
1. Loading a trained MFRouter
2. Routing queries using matrix factorization

MFRouter is effective for:
- Capturing query-model interaction patterns
- Collaborative filtering-style routing